# lung_even_consistency

Create a capped (evened) version of the `lung.h5ad` dataset by limiting the maximum number of cells per annotated class. Saves sampled AnnData, CSV counts, and bar charts. Parameters are configurable in the first code cell.

In [1]:
# Parameters
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = 'lung.h5ad'
OUT_DIR = 'lung_even_consistency_results'
os.makedirs(OUT_DIR, exist_ok=True)
# cap: maximum number of cells to keep per cell-type (default suggestion: 1000)
max_per_type = 1000
# drop very rare classes with fewer than this many cells
min_cells_per_type = 20
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [2]:
# Helper: detect annotation column (common names then first categorical)
def detect_annotation_column(adata):
    candidates = ['cell_type', 'celltype', 'cell_type1', 'cell_ontology_class', 'annotation']
    for c in candidates:
        if c in adata.obs.columns:
            return c
    # fallback: first categorical/object column
    for c in adata.obs.columns:
        if pd.api.types.is_categorical_dtype(adata.obs[c]) or adata.obs[c].dtype == object:
            return c
    raise ValueError('No suitable annotation column found in adata.obs')

# Load data (safe load then optional slicing if needed)
adata = sc.read_h5ad(DATA_PATH)
print('Loaded', DATA_PATH, 'with', adata.n_obs, 'cells and', adata.n_vars, 'genes')
ann_col = detect_annotation_column(adata)
print('Using annotation column:', ann_col)

# original counts and plot
orig_counts = adata.obs[ann_col].value_counts().sort_values(ascending=False)
print('Top classes (original):')
print(orig_counts.head(20))
plt.figure(figsize=(10,6))
sns.barplot(x=orig_counts.values, y=orig_counts.index, palette='viridis')
plt.xlabel('Cell counts')
plt.title('Original counts per class')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'lung_even_original_counts.png'), dpi=150)
plt.close()
orig_counts.to_csv(os.path.join(OUT_DIR, 'lung_even_original_counts.csv'), header=['count'])

Loaded lung.h5ad with 584944 cells and 27402 genes
Using annotation column: cell_type
Top classes (original):
cell_type
respiratory basal cell                                  80113
alveolar macrophage                                     78816
pulmonary alveolar type 2 cell                          62405
club cell                                               36023
nasal mucosa goblet cell                                35833
multiciliated columnar cell of tracheobronchial tree    35225
CD8-positive, alpha-beta T cell                         29074
elicited macrophage                                     28223
capillary endothelial cell                              23205
CD4-positive, alpha-beta T cell                         21285
classical monocyte                                      17695
natural killer cell                                     16978
vein endothelial cell                                   12975
alveolar adventitial fibroblast                         10321
CD1c-positiv

/tmp/ipykernel_32232/3155349858.py:24: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=orig_counts.values, y=orig_counts.index, palette='viridis')


In [3]:
# Build capped (even) sample
labels = adata.obs[ann_col].astype(str)
counts = labels.value_counts()
# keep only sufficiently represented classes
keep_labels = counts[counts >= min_cells_per_type].index.tolist()
print(f'Keeping {len(keep_labels)} classes with >= {min_cells_per_type} cells')
sampled_idx = []
for lbl in keep_labels:
    idxs = np.where(labels.values == lbl)[0]
    n_keep = min(len(idxs), max_per_type)
    chosen = np.random.choice(idxs, size=n_keep, replace=False)
    sampled_idx.extend(chosen.tolist())

# create sampled AnnData (preserve ordering)
sampled_idx = np.array(sampled_idx, dtype=int)
sampled_adata = adata[sampled_idx].copy()
print('Sampled dataset has', sampled_adata.n_obs, 'cells across', len(sampled_adata.obs[ann_col].unique()), 'classes')

# save sampled AnnData
sampled_path = os.path.join(OUT_DIR, 'lung_even.h5ad')
sampled_adata.write(sampled_path)
print('Wrote sampled AnnData to', sampled_path)

# save new counts and plot
new_counts = sampled_adata.obs[ann_col].value_counts().sort_values(ascending=False)
new_counts.to_csv(os.path.join(OUT_DIR, 'lung_even_counts.csv'), header=['count'])
plt.figure(figsize=(10,6))
sns.barplot(x=new_counts.values, y=new_counts.index, palette='magma')
plt.xlabel('Cell counts (capped)')
plt.title('Evened counts per class (capped)')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'lung_even_cell_counts.png'), dpi=150)
plt.close()

Keeping 50 classes with >= 20 cells
Sampled dataset has 41249 cells across 50 classes
Wrote sampled AnnData to lung_even_consistency_results/lung_even.h5ad


/tmp/ipykernel_32232/748152030.py:28: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=new_counts.values, y=new_counts.index, palette='magma')


## Rationale and next steps

- Default `max_per_type = 1000` is a reasonable starting point: it keeps hundreds to a thousand cells per class (reduces class imbalance) while retaining class representation for typical single-cell analyses.
- If your dataset has many rare classes, increase `min_cells_per_type` to drop extremely small/unstable classes, or reduce `max_per_type` if memory/time is a concern.
- If you want, I can now run this notebook to produce the sampled file and the plots, or adjust `max_per_type`/`min_cells_per_type` before running.

## Model Training & Evaluation on Balanced Dataset

Compare model performance between original (imbalanced) and evened (balanced) datasets to see if large cell counts skew results.

In [4]:

import torch
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Configuration for model training
print("=" * 80)
print("MODEL TRAINING & EVALUATION ON BALANCED (EVENED) DATASET")
print("=" * 80)

# Device configuration
def get_device():
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
    elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
        device = torch.device("mps")
        print("Using Apple Metal Performance Shaders (MPS)")
    else:
        device = torch.device("cpu")
        print("Using CPU (no GPU detected)")
    return device

device = get_device()

# Model and data parameters (same as lung_consistency_evaluation)
c2s_model_name = "vandijklab/C2S-Pythia-410m-cell-type-conditioned-cell-generation"
batch_size = 8
top_k = 100

# Train/val/test split
train_frac = 0.70
val_frac = 0.15
test_frac = 0.15

# Multiple splits for evaluation
n_splits = 5

print(f"\n📋 Model Configuration:")
print(f"  Model: {c2s_model_name.split('/')[-1]}")
print(f"  Device: {device}")
print(f"  Train/Val/Test split: {train_frac}/{val_frac}/{test_frac}")
print(f"  Number of evaluation splits: {n_splits}")


MODEL TRAINING & EVALUATION ON BALANCED (EVENED) DATASET
Using CUDA GPU: NVIDIA GeForce RTX 5080

📋 Model Configuration:
  Model: C2S-Pythia-410m-cell-type-conditioned-cell-generation
  Device: cuda
  Train/Val/Test split: 0.7/0.15/0.15
  Number of evaluation splits: 5


In [5]:

# Helper functions (same as lung_consistency_evaluation)
def cell_to_text(cell_vector, gene_names, top_k=100):
    """Convert cell expression to top-k gene names."""
    if hasattr(cell_vector, "toarray"):
        vec = cell_vector.toarray().flatten()
    else:
        vec = np.asarray(cell_vector).flatten()
    
    if vec.size == 0:
        return ""
    
    top_idx = np.argsort(vec)[-top_k:][::-1]
    top_genes = [str(gene_names[i]) for i in top_idx if gene_names[i]]
    return " ".join(top_genes)


def filter_empty(texts, ids, labels):
    """Remove cells with empty gene text."""
    keep = [i for i, t in enumerate(texts) if isinstance(t, str) and t.strip() != ""]
    return [texts[i] for i in keep], np.array(ids)[keep].tolist(), np.array(labels)[keep].tolist()


def stratified_train_val_test_indices(labels, train_frac, val_frac, test_frac, random_state=None):
    """Return train, val, test indices stratified by labels."""
    idx = np.arange(len(labels))
    idx_trainval, idx_test, y_trainval, y_test = train_test_split(
        idx, labels, test_size=test_frac, stratify=labels, random_state=random_state
    )
    relative_val_frac = val_frac / (train_frac + val_frac)
    idx_train, idx_val, y_train, y_val = train_test_split(
        idx_trainval, y_trainval, test_size=relative_val_frac, stratify=y_trainval, random_state=random_state
    )
    return idx_train, idx_val, idx_test


print("✅ Helper functions defined")


✅ Helper functions defined


In [6]:

# Prepare data from balanced (evened) dataset
print("\n📊 Preparing data from evened dataset...")

# Load the evened dataset we just created
evened_path = os.path.join(OUT_DIR, 'lung_even.h5ad')
if not os.path.exists(evened_path):
    print(f"⚠️  Evened dataset not found at {evened_path}")
    print("Creating it now...")
    # Re-run the sampling from earlier cells
else:
    adata_even = sc.read_h5ad(evened_path)
    print(f"Loaded evened dataset: {adata_even.n_obs} cells, {adata_even.n_vars} genes")
    
    # Prepare texts and labels
    cell_texts = [cell_to_text(adata_even.X[i], adata_even.var_names, top_k=top_k) 
                  for i in range(adata_even.n_obs)]
    cell_texts, cell_ids, cell_labels = filter_empty(cell_texts, adata_even.obs_names, 
                                                       adata_even.obs[ann_col].astype(str).values)
    
    cell_labels = np.array(cell_labels)
    cell_ids = np.array(cell_ids)
    
    print(f"✅ Prepared {len(cell_texts)} cells")
    print(f"   Unique cell types: {len(np.unique(cell_labels))}")
    print(f"\n📊 Class distribution (evened dataset):")
    evened_dist = pd.Series(cell_labels).value_counts()
    for idx, (cell_type, count) in enumerate(evened_dist.head(10).items(), 1):
        pct = count / len(cell_labels) * 100
        print(f"   {idx:2d}. {cell_type:40s}: {count:5d} cells ({pct:5.1f}%)")
    if len(evened_dist) > 10:
        print(f"   ... and {len(evened_dist) - 10} more cell types")



📊 Preparing data from evened dataset...
Loaded evened dataset: 41249 cells, 27402 genes
✅ Prepared 41249 cells
   Unique cell types: 50

📊 Class distribution (evened dataset):
    1. respiratory basal cell                  :  1000 cells (  2.4%)
    2. alveolar macrophage                     :  1000 cells (  2.4%)
    3. pulmonary alveolar type 2 cell          :  1000 cells (  2.4%)
    4. club cell                               :  1000 cells (  2.4%)
    5. nasal mucosa goblet cell                :  1000 cells (  2.4%)
    6. multiciliated columnar cell of tracheobronchial tree:  1000 cells (  2.4%)
    7. CD8-positive, alpha-beta T cell         :  1000 cells (  2.4%)
    8. elicited macrophage                     :  1000 cells (  2.4%)
    9. capillary endothelial cell              :  1000 cells (  2.4%)
   10. CD4-positive, alpha-beta T cell         :  1000 cells (  2.4%)
   ... and 40 more cell types


In [7]:

# Load cell2sentence model
print("\n🔄 Loading cell2sentence model...")

from transformers import AutoTokenizer, AutoModel

try:
    print(f"Loading: {c2s_model_name}")
    tokenizer = AutoTokenizer.from_pretrained(c2s_model_name)
    model = AutoModel.from_pretrained(c2s_model_name, output_hidden_states=False)
    model.to(device)
    model.eval()
    print("✅ Model loaded successfully!")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    raise

print(f"Model device: {device}")



🔄 Loading cell2sentence model...
Loading: vandijklab/C2S-Pythia-410m-cell-type-conditioned-cell-generation


Loading weights: 100%|██████████| 291/291 [00:00<00:00, 2762.13it/s, Materializing param=layers.23.post_attention_layernorm.weight] 
GPTNeoXModel LOAD REPORT from: vandijklab/C2S-Pythia-410m-cell-type-conditioned-cell-generation
Key              | Status     |  | 
-----------------+------------+--+-
embed_out.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded successfully!
Model device: cuda


In [8]:

# Generate embeddings for evened dataset
def generate_embeddings(texts, tokenizer, model, device, batch_size=8):
    """Generate embeddings for a batch of texts."""
    embeddings = []
    
    print(f"Generating embeddings for {len(texts)} cells...")
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]
        
        try:
            inputs = tokenizer(
                batch_texts, 
                return_tensors="pt", 
                padding=True, 
                truncation=True,
                max_length=512
            ).to(device)
            
            with torch.no_grad():
                outputs = model(**inputs)
            
            last_hidden = outputs.last_hidden_state.cpu().numpy()
            
            for j in range(last_hidden.shape[0]):
                embedding = last_hidden[j].mean(axis=0)
                embeddings.append(embedding)
        except Exception as e:
            print(f"Error processing batch {i//batch_size}: {e}")
            embeddings.extend([np.zeros(last_hidden.shape[-1])] * len(batch_texts))
    
    return np.array(embeddings)


print("\n🔄 Generating embeddings for evened dataset (single pass)...")
evened_embeddings = generate_embeddings(cell_texts, tokenizer, model, device, batch_size)
print(f"✅ Evened embeddings shape: {evened_embeddings.shape}")

# Save embeddings
evened_emb_df = pd.DataFrame(evened_embeddings, index=cell_ids)
evened_emb_path = os.path.join(OUT_DIR, "lung_even_embeddings.csv")
evened_emb_df.to_csv(evened_emb_path)
print(f"✅ Embeddings saved to: {evened_emb_path}")



🔄 Generating embeddings for evened dataset (single pass)...
Generating embeddings for 41249 cells...


100%|██████████| 5157/5157 [10:48<00:00,  7.95it/s]


✅ Evened embeddings shape: (41249, 1024)
✅ Embeddings saved to: lung_even_consistency_results/lung_even_embeddings.csv


In [ ]:

# Train and evaluate classifiers on evened dataset (multiple splits)
print("\n" + "=" * 80)
print("MULTI-SPLIT EVALUATION ON EVENED DATASET")
print("=" * 80)

# Encode labels
le_even = LabelEncoder()
evened_labels_encoded = le_even.fit_transform(cell_labels)

print(f"\nClasses ({len(le_even.classes_)}): {le_even.classes_[:5]}...")

# Container for results across splits
evened_records = []

print(f"\nTraining across {n_splits} stratified splits (train/val/test)...\n")

for run in range(n_splits):
    rs = 42 + run
    train_idx, val_idx, test_idx = stratified_train_val_test_indices(
        evened_labels_encoded, train_frac, val_frac, test_frac, random_state=rs
    )
    
    X_train = evened_embeddings[train_idx]
    y_train = evened_labels_encoded[train_idx]
    
    X_val = evened_embeddings[val_idx]
    y_val = evened_labels_encoded[val_idx]
    
    X_test = evened_embeddings[test_idx]
    y_test = evened_labels_encoded[test_idx]
    
    # Logistic Regression
    lr = LogisticRegression(max_iter=2000, random_state=rs, n_jobs=-1, C=0.1, 
                            solver='lbfgs', class_weight='balanced')
    lr.fit(X_train, y_train)
    val_pred_lr = lr.predict(X_val)
    test_pred_lr = lr.predict(X_test)
    
    val_acc_lr = accuracy_score(y_val, val_pred_lr)
    test_acc_lr = accuracy_score(y_test, test_pred_lr)
    
    # Random Forest
    rf = RandomForestClassifier(
        n_estimators=200, max_depth=15, min_samples_split=10, min_samples_leaf=5,
        max_samples=0.8, max_features='sqrt', random_state=rs, n_jobs=-1, class_weight='balanced'
    )
    rf.fit(X_train, y_train)
    val_pred_rf = rf.predict(X_val)
    test_pred_rf = rf.predict(X_test)
    
    val_acc_rf = accuracy_score(y_val, val_pred_rf)
    test_acc_rf = accuracy_score(y_test, test_pred_rf)
    
    evened_records.append({
        'run': run,
        'lr_val_acc': val_acc_lr,
        'lr_test_acc': test_acc_lr,
        'rf_val_acc': val_acc_rf,
        'rf_test_acc': test_acc_rf,
        'n_train': len(train_idx),
        'n_val': len(val_idx),
        'n_test': len(test_idx)
    })
    
    print(f"Run {run}: LR val={val_acc_lr:.4f}, test={test_acc_lr:.4f} | RF val={val_acc_rf:.4f}, test={test_acc_rf:.4f}")

# Summarize results
evened_results_df = pd.DataFrame(evened_records)
evened_summary_path = os.path.join(OUT_DIR, "lung_even_multi_split_results.csv")
evened_results_df.to_csv(evened_summary_path, index=False)

print(f"\n✅ Evened results saved to: {evened_summary_path}")
print('\nSummary (mean ± std):')
for col in ['lr_test_acc', 'rf_test_acc', 'lr_val_acc', 'rf_val_acc']:
    mean_val = evened_results_df[col].mean()
    std_val = evened_results_df[col].std()
    print(f"  {col}: {mean_val:.4f} ± {std_val:.4f}")



MULTI-SPLIT EVALUATION ON EVENED DATASET

Classes (50): ['B cell' 'CD1c-positive myeloid dendritic cell'
 'CD4-positive, alpha-beta T cell' 'CD8-positive, alpha-beta T cell'
 'T cell']...

Training across 5 stratified splits (train/val/test)...

Run 0: LR val=0.8180, test=0.8130 | RF val=0.6382, test=0.6409
Run 1: LR val=0.8176, test=0.8198 | RF val=0.6353, test=0.6377


In [ ]:

# Comparison: Original vs Evened Dataset
print("\n" + "=" * 80)
print("COMPARISON: ORIGINAL vs EVENED DATASET")
print("=" * 80)

# Load original results (if available)
original_results_path = '/home/hugolab/cell2sentence/c2s-prediction-model/lung_consistency_results/lung_multi_split_results.csv'

try:
    original_results_df = pd.read_csv(original_results_path)
    print("\n✅ Loaded original (imbalanced) dataset results")
    
    # Create comparison table
    comparison_data = []
    
    for metric in ['lr_test_acc', 'rf_test_acc']:
        orig_mean = original_results_df[metric].mean()
        orig_std = original_results_df[metric].std()
        even_mean = evened_results_df[metric].mean()
        even_std = evened_results_df[metric].std()
        diff = even_mean - orig_mean
        pct_change = (diff / orig_mean) * 100
        
        comparison_data.append({
            'Metric': metric,
            'Original (Imbalanced)': f"{orig_mean:.4f} ± {orig_std:.4f}",
            'Evened (Balanced)': f"{even_mean:.4f} ± {even_std:.4f}",
            'Difference': f"{diff:+.4f}",
            'Pct Change': f"{pct_change:+.2f}%"
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    print("\n" + comparison_df.to_string(index=False))
    
    # Save comparison
    comparison_path = os.path.join(OUT_DIR, "lung_imbalanced_vs_balanced_comparison.csv")
    comparison_df.to_csv(comparison_path, index=False)
    print(f"\n✅ Comparison saved to: {comparison_path}")
    
    # Visualize comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))As I am currently based in California, having this information would be incredibly helpful as I begin planning flights and housing.

Thank you so much for your time and help — I really appreciate it.

Best regards,
Leah Shin
    
    # LR comparison
    ax = axes[0]
    orig_lr = original_results_df['lr_test_acc'].values
    even_lr = evened_results_df['lr_test_acc'].values
    
    x = np.arange(len(orig_lr))
    width = 0.35
    ax.bar(x - width/2, orig_lr, width, label='Original (Imbalanced)', alpha=0.8, color='steelblue')
    ax.bar(x + width/2, even_lr, width, label='Evened (Balanced)', alpha=0.8, color='coral')
    
    ax.set_ylabel('Test Accuracy', fontsize=11)
    ax.set_xlabel('Split Number', fontsize=11)
    ax.set_title('Logistic Regression - Test Accuracy Comparison', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.legend()
    ax.set_ylim([0, 1.0])
    ax.grid(axis='y', alpha=0.3)
    
    # RF comparison
    ax = axes[1]
    orig_rf = original_results_df['rf_test_acc'].values
    even_rf = evened_results_df['rf_test_acc'].values
    
    ax.bar(x - width/2, orig_rf, width, label='Original (Imbalanced)', alpha=0.8, color='steelblue')
    ax.bar(x + width/2, even_rf, width, label='Evened (Balanced)', alpha=0.8, color='coral')
    
    ax.set_ylabel('Test Accuracy', fontsize=11)
    ax.set_xlabel('Split Number', fontsize=11)
    ax.set_title('Random Forest - Test Accuracy Comparison', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.legend()
    ax.set_ylim([0, 1.0])
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    comp_fig_path = os.path.join(OUT_DIR, "lung_comparison_accuracy.png")
    plt.savefig(comp_fig_path, dpi=200, bbox_inches='tight')
    print(f"✅ Comparison figure saved to: {comp_fig_path}")
    plt.show()
    
except FileNotFoundError:
    print(f"⚠️  Original results not found at {original_results_path}")
    print("Run lung_consistency_evaluation.ipynb first to generate comparison data")
except Exception as e:
    print(f"⚠️  Could not load original results: {e}")


In [ ]:

# Final Summary Report
print("\n" + "=" * 80)
print("BALANCED DATASET EVALUATION - SUMMARY REPORT")
print("=" * 80)

summary_text = f"""
Dataset Configuration:
  Original (Imbalanced):   {original_results_df.iloc[0]['n_train'] + original_results_df.iloc[0]['n_val'] + original_results_df.iloc[0]['n_test']:,} cells
  Evened (Balanced):       {evened_results_df.iloc[0]['n_train'] + evened_results_df.iloc[0]['n_val'] + evened_results_df.iloc[0]['n_test']:,} cells
  Max per class (cap):     {max_per_type}
  Min cells per class:     {min_cells_per_type}

Logistic Regression Performance:
  Original:  {original_results_df['lr_test_acc'].mean():.4f} ± {original_results_df['lr_test_acc'].std():.4f}
  Evened:    {evened_results_df['lr_test_acc'].mean():.4f} ± {evened_results_df['lr_test_acc'].std():.4f}
  Change:    {(evened_results_df['lr_test_acc'].mean() - original_results_df['lr_test_acc'].mean()):+.4f}

Random Forest Performance:
  Original:  {original_results_df['rf_test_acc'].mean():.4f} ± {original_results_df['rf_test_acc'].std():.4f}
  Evened:    {evened_results_df['rf_test_acc'].mean():.4f} ± {evened_results_df['rf_test_acc'].std():.4f}
  Change:    {(evened_results_df['rf_test_acc'].mean() - original_results_df['rf_test_acc'].mean()):+.4f}

Interpretation:
{"✓ Class imbalance DOES skew results (evened > original)" if evened_results_df['lr_test_acc'].mean() > original_results_df['lr_test_acc'].mean() else "✗ No significant impact from class imbalance"}
"""

print(summary_text)

# Save summary
summary_path = os.path.join(OUT_DIR, "lung_balanced_vs_imbalanced_summary.txt")
with open(summary_path, 'w') as f:
    f.write(summary_text)

print(f"✅ Summary saved to: {summary_path}\n")
print("=" * 80)
